In [1]:
"""
DiscountR – LangGraph skeleton (Sprint 2, Fase A/B)
Doel: aantonen dat State, Nodes, Conditional Edges en loops werken.
Vul stap voor stap aan met echte API-calls (Maps, SupermarktConnector).
"""

from __future__ import annotations
from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict
from operator import add

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END


# ---------- 1. Schemas (Pydantic – gestructureerde output) ----------

class Boodschap(BaseModel):
    naam: str = Field(description="Productnaam, bv. 'volle melk'")
    hoeveelheid: int = Field(default=1, ge=1)
    eenheid: str = Field(default="stuk")

class BoodschappenLijst(BaseModel):
    items: list[Boodschap]

class Locatie(BaseModel):
    adres: str
    lat: float
    lng: float


# ---------- 2. Gedeelde State ----------
# Annotated[list, add] => lijsten worden gemerged ipv overschreven (handig voor logs)

class AgentState(TypedDict, total=False):
    user_input: str
    locatie_hint: Optional[str]
    locatie: Optional[Locatie]
    boodschappen: Optional[BoodschappenLijst]
    fouten: Annotated[list[str], add]
    retries: int
    advies: Optional[str]


# ---------- 3. Nodes ----------

def node_user_input(state: AgentState) -> AgentState:
    # In productie: hier komt input van UI/chat
    print("[1] user_input ontvangen")
    return {"retries": state.get("retries", 0)}


def node_locatie(state: AgentState) -> AgentState:
    print("[2] locatie bepalen (stub – vervang door Google Maps geocoding)")
    # TODO: googlemaps.Client(key=...).geocode(state['locatie_hint'])
    return {
        "locatie": Locatie(adres="Den Haag, NL", lat=52.0705, lng=4.3007)
    }


def node_parse_lijst(state: AgentState) -> AgentState:
    """Stap 3 – LLM met structured output. Hier eerst een stub die kan falen."""
    print("[3] boodschappenlijst parsen")
    text = state.get("user_input", "")

    # TODO: vervang door:
    # llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(BoodschappenLijst)
    # lijst = llm.invoke(f"Parse deze boodschappenlijst: {text}")

    if not text.strip():
        # Forceer een faal om de loop te demonstreren
        return {"fouten": ["lege input – kan niet parsen"]}

    # Dummy parse
    items = [Boodschap(naam=w.strip()) for w in text.split(",") if w.strip()]
    return {"boodschappen": BoodschappenLijst(items=items)}


def node_advies(state: AgentState) -> AgentState:
    print("[8/9] advies genereren")
    n = len(state["boodschappen"].items)
    return {"advies": f"Gevonden {n} producten in {state['locatie'].adres}"}


# ---------- 4. Validatie + conditional edges (LOOPS) ----------

MAX_RETRIES = 2

def valideer_parse(state: AgentState) -> Literal["ok", "retry", "geef_op"]:
    if state.get("boodschappen") and state["boodschappen"].items:
        return "ok"
    if state.get("retries", 0) >= MAX_RETRIES:
        return "geef_op"
    return "retry"


def node_retry(state: AgentState) -> AgentState:
    print(f"   ↻ retry (poging {state.get('retries', 0) + 1})")
    # Simuleer dat we nu wel input hebben (in productie: vraag user opnieuw)
    return {
        "retries": state.get("retries", 0) + 1,
        "user_input": "melk, brood, kaas",
    }


# ---------- 5. Graph compositie ----------

def build_graph():
    g = StateGraph(AgentState)

    g.add_node("user_input", node_user_input)
    g.add_node("locatie", node_locatie)
    g.add_node("parse_lijst", node_parse_lijst)
    g.add_node("retry_input", node_retry)
    g.add_node("advies", node_advies)

    g.add_edge(START, "user_input")
    g.add_edge("user_input", "locatie")
    g.add_edge("locatie", "parse_lijst")

    # Conditional edge = de LOOP uit jullie requirements
    g.add_conditional_edges(
        "parse_lijst",
        valideer_parse,
        {
            "ok": "advies",
            "retry": "retry_input",
            "geef_op": END,
        },
    )
    g.add_edge("retry_input", "parse_lijst")
    g.add_edge("advies", END)

    return g.compile()


# ---------- 6. Run + visualisatie ----------

if __name__ == "__main__":
    app = build_graph()

    # Mermaid export voor je verslag
    print("\n=== MERMAID DIAGRAM ===")
    print(app.get_graph().draw_mermaid())
    print("=======================\n")

    # Test 1: lege input -> moet retry triggeren
    print("--- RUN 1: lege input (loop test) ---")
    result = app.invoke({"user_input": "", "locatie_hint": "Den Haag"})
    print("Resultaat:", result.get("advies"))
    print("Fouten:", result.get("fouten"))

    # Test 2: directe input
    print("\n--- RUN 2: directe input ---")
    result = app.invoke({"user_input": "melk, brood, appels", "locatie_hint": "Den Haag"})
    print("Resultaat:", result.get("advies"))



=== MERMAID DIAGRAM ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	user_input(user_input)
	locatie(locatie)
	parse_lijst(parse_lijst)
	retry_input(retry_input)
	advies(advies)
	__end__([<p>__end__</p>]):::last
	__start__ --> user_input;
	locatie --> parse_lijst;
	parse_lijst -. &nbsp;geef_op&nbsp; .-> __end__;
	parse_lijst -. &nbsp;ok&nbsp; .-> advies;
	parse_lijst -. &nbsp;retry&nbsp; .-> retry_input;
	retry_input --> parse_lijst;
	user_input --> locatie;
	advies --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


--- RUN 1: lege input (loop test) ---
[1] user_input ontvangen
[2] locatie bepalen (stub – vervang door Google Maps geocoding)
[3] boodschappenlijst parsen
   ↻ retry (poging 1)
[3] boodschappenlijst parsen
[8/9] advies genereren
Resultaat: Gevonden 3 producten in Den Haag, NL
Fouten: ['lege input – kan niet parsen']

--- RUN 2: directe input ---
[1] us